# Logistic Regression From Scratch — MNIST Phase 1

This notebook lets you choose any feature extraction method:

- `flatten`
- `hog`
- `pca`

It also tunes:

- learning rate for all methods
- PCA components only when `method = "pca"`

Removed sections:

- Linearity check
- Visual linear decision boundary
- Visual sample prediction

## 0. Imports

`preprocessing.py` must be in the same folder as this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import importlib

import preprocessing
importlib.reload(preprocessing)

from preprocessing import preprocess_mnist

## 1. Choose Method and Settings

Change only this cell when you want to test another method.

In [ ]:
target_digit = 0

# Choose one:
# method = "flatten"
# method = "hog"
# method = "pca"
method = "pca"

# Learning rates are tested for all methods
learning_rates = [0.001, 0.005, 0.01, 0.05, 0.1]

# PCA components are used only when method = "pca"
pca_components_list = [50, 75, 100, 150, 200]

epochs = 2000
print_every = 500
val_ratio = 0.15
threshold = 0.5

## 2. Logistic Regression Functions

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def compute_loss(y_true, y_pred):
    eps = 1e-8
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(
        y_true * np.log(y_pred) +
        (1 - y_true) * np.log(1 - y_pred)
    )


def predict_probability(X, W, b):
    return sigmoid(np.dot(X, W) + b)


def predict_binary(X, W, b, threshold=0.5):
    return (predict_probability(X, W, b) >= threshold).astype(int)


def convert_to_class_1_class_2(predictions):
    return np.where(predictions == 1, 1, 2)

## 3. Evaluation Metrics

In [ ]:
def evaluate(y_true, y_pred):
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))

    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN
    }

## 4. Training Function

Training uses only the training set.

Validation is used to keep the best epoch.

In [ ]:
def train_logistic_regression(
    X_train,
    y_train,
    X_val,
    y_val,
    learning_rate=0.05,
    epochs=2000,
    print_every=500
):
    m, n = X_train.shape

    W = np.zeros(n)
    b = 0.0

    train_losses = []
    val_losses = []

    best_val_loss = float("inf")
    best_W = W.copy()
    best_b = b
    best_epoch = 0

    for epoch in range(epochs + 1):
        y_train_prob = predict_probability(X_train, W, b)
        train_loss = compute_loss(y_train, y_train_prob)

        error = y_train_prob - y_train
        dW = (X_train.T @ error) / m
        db = error.mean()

        W -= learning_rate * dW
        b -= learning_rate * db

        y_val_prob = predict_probability(X_val, W, b)
        val_loss = compute_loss(y_val, y_val_prob)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_W = W.copy()
            best_b = b
            best_epoch = epoch

        if print_every and epoch % print_every == 0:
            print(
                f"Epoch {epoch:04d} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_loss:.4f}"
            )

    return {
        "W": best_W,
        "b": best_b,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "train_losses": train_losses,
        "val_losses": val_losses
    }

## 5. Tuning Logic

Rules:

- If `method = "pca"`:
  - tune PCA components and learning rate
- If `method = "flatten"` or `method = "hog"`:
  - tune learning rate only

In [ ]:
if method not in ["flatten", "hog", "pca"]:
    raise ValueError("method must be 'flatten', 'hog', or 'pca'")

if method == "pca":
    feature_settings = pca_components_list
else:
    feature_settings = [None]

tuning_results = []
best_result = None

for pca_components in feature_settings:
    print("\n" + "=" * 70)

    if method == "pca":
        print(f"METHOD = PCA | PCA COMPONENTS = {pca_components}")
    else:
        print(f"METHOD = {method.upper()}")

    print("=" * 70)

    X_train, y_train, X_val, y_val, X_test, y_test, mean, std = preprocess_mnist(
        target_digit=target_digit,
        method=method,
        pca_components=100 if pca_components is None else pca_components,
        val_ratio=val_ratio
    )

    print("X_train:", X_train.shape)
    print("X_val  :", X_val.shape)
    print("X_test :", X_test.shape)

    n_pos = np.sum(y_train == 1)
    n_neg = np.sum(y_train == 0)
    print(f"Class 1 samples: {n_pos}")
    print(f"Class 2 samples: {n_neg}")
    print(f"Imbalance ratio: 1 : {n_neg // n_pos}")

    for learning_rate in learning_rates:
        print("\n" + "-" * 70)

        if method == "pca":
            print(f"Training method=pca, PCA={pca_components}, LR={learning_rate}")
        else:
            print(f"Training method={method}, LR={learning_rate}")

        print("-" * 70)

        model = train_logistic_regression(
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            learning_rate=learning_rate,
            epochs=epochs,
            print_every=print_every
        )

        y_val_pred = predict_binary(X_val, model["W"], model["b"], threshold=threshold)
        val_metrics = evaluate(y_val, y_val_pred)

        result = {
            "target_digit": target_digit,
            "method": method,
            "pca_components": pca_components,
            "learning_rate": learning_rate,
            "threshold": threshold,
            "best_epoch": model["best_epoch"],
            "best_val_loss": model["best_val_loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "model": model
        }

        tuning_results.append(result)

        print("\nValidation Results")
        print("==================")
        print(f"Accuracy : {val_metrics['accuracy']:.4f}")
        print(f"Precision: {val_metrics['precision']:.4f}")
        print(f"Recall   : {val_metrics['recall']:.4f}")
        print(f"F1 Score : {val_metrics['f1']:.4f}")

        if best_result is None or result["val_f1"] > best_result["val_f1"]:
            best_result = result

print("\nTuning finished.")

## 6. Tuning Results Table

In [ ]:
tuning_results_sorted = sorted(
    tuning_results,
    key=lambda r: r["val_f1"],
    reverse=True
)

print("TOP TUNING RESULTS")
print("==================")

if method == "pca":
    print(
        f"{'Rank':<5} {'Method':<10} {'PCA':<6} {'LR':<10} {'Epoch':<8} "
        f"{'Val Loss':<10} {'Val Acc':<10} {'Val Prec':<10} {'Val Rec':<10} {'Val F1':<10}"
    )
else:
    print(
        f"{'Rank':<5} {'Method':<10} {'LR':<10} {'Epoch':<8} "
        f"{'Val Loss':<10} {'Val Acc':<10} {'Val Prec':<10} {'Val Rec':<10} {'Val F1':<10}"
    )

for rank, r in enumerate(tuning_results_sorted, start=1):
    if method == "pca":
        print(
            f"{rank:<5} "
            f"{r['method']:<10} "
            f"{r['pca_components']:<6} "
            f"{r['learning_rate']:<10} "
            f"{r['best_epoch']:<8} "
            f"{r['best_val_loss']:<10.4f} "
            f"{r['val_accuracy']:<10.4f} "
            f"{r['val_precision']:<10.4f} "
            f"{r['val_recall']:<10.4f} "
            f"{r['val_f1']:<10.4f}"
        )
    else:
        print(
            f"{rank:<5} "
            f"{r['method']:<10} "
            f"{r['learning_rate']:<10} "
            f"{r['best_epoch']:<8} "
            f"{r['best_val_loss']:<10.4f} "
            f"{r['val_accuracy']:<10.4f} "
            f"{r['val_precision']:<10.4f} "
            f"{r['val_recall']:<10.4f} "
            f"{r['val_f1']:<10.4f}"
        )

print("\nBEST CONFIGURATION")
print("==================")
print("Method        :", best_result["method"])

if best_result["method"] == "pca":
    print("PCA components:", best_result["pca_components"])

print("Learning rate :", best_result["learning_rate"])
print("Best epoch    :", best_result["best_epoch"])
print("Validation F1 :", round(best_result["val_f1"], 4))

## 7. Final Test Evaluation

The test set is used only here after the best validation configuration is selected.

In [ ]:
best_method = best_result["method"]
best_learning_rate = best_result["learning_rate"]
best_pca_components = best_result["pca_components"]
best_model = best_result["model"]

X_train, y_train, X_val, y_val, X_test, y_test, mean, std = preprocess_mnist(
    target_digit=target_digit,
    method=best_method,
    pca_components=100 if best_pca_components is None else best_pca_components,
    val_ratio=val_ratio
)

W = best_model["W"]
b = best_model["b"]

y_test_pred = predict_binary(X_test, W, b, threshold=threshold)
final_predictions = convert_to_class_1_class_2(y_test_pred)

test_metrics = evaluate(y_test, y_test_pred)

accuracy = test_metrics["accuracy"]
precision = test_metrics["precision"]
recall = test_metrics["recall"]
f1 = test_metrics["f1"]
TP = test_metrics["TP"]
TN = test_metrics["TN"]
FP = test_metrics["FP"]
FN = test_metrics["FN"]

print("FINAL TEST RESULTS")
print("==================")
print(f"Target digit   : {target_digit}")
print(f"Class 1        : digit {target_digit}")
print(f"Class 2        : all other digits")
print(f"Feature method : {best_method}")

if best_method == "pca":
    print(f"PCA components : {best_pca_components}")

print(f"Learning rate  : {best_learning_rate}")
print(f"Best val epoch : {best_result['best_epoch']}")
print(f"Threshold      : {threshold}")
print()
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print()
print("Confusion Matrix  (Rows=Actual | Cols=Predicted)")
print("              Pred C2   Pred C1")
print(f"  Actual C2     {TN:5d}    {FP:5d}")
print(f"  Actual C1     {FN:5d}    {TP:5d}")
print()
print("First 20 predictions (Class 1 / Class 2):")
print(final_predictions[:20])

## 8. Loss Curve for Selected Model

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(best_model["train_losses"], label="Training Loss", lw=1.5)
plt.plot(best_model["val_losses"], label="Validation Loss", lw=1.5)
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")

if best_method == "pca":
    title = f"Best Model Loss Curve | Method=PCA, PCA={best_pca_components}, LR={best_learning_rate}"
else:
    title = f"Best Model Loss Curve | Method={best_method}, LR={best_learning_rate}"

plt.title(title)
plt.legend()
plt.grid(True)
plt.show()

## 9. Final Results Row

In [ ]:
results_row = {
    "model": "Logistic Regression",
    "target_digit": target_digit,
    "method": best_method,
    "pca_components": None if best_pca_components is None else int(best_pca_components),
    "learning_rate": float(best_learning_rate),
    "best_epoch": int(best_result["best_epoch"]),
    "threshold": float(threshold),
    "accuracy": round(float(accuracy), 4),
    "precision": round(float(precision), 4),
    "recall": round(float(recall), 4),
    "f1": round(float(f1), 4),
}

print(results_row)